# Travelling Salesman Problem via QUBO + QAOA

This notebook compares three ways to represent the same TSP QUBO.


## Problem size

The TSP uses **n² qubits** — 8 cities means a **64-qubit QUBO**. Even with partitioning, each sub-problem is a full QAOA optimization. QoroService **parallelizes the partitioned sub-problems** and handles deeper circuits with Maestro.

| Cities | Qubits | Brute-force permutations |
|--------|--------|-------------------------|
| 3 | 9 | 2 |
| 4 | 16 | 6 |
| 5 | 25 | 24 |
| 8 | 64 | 5,040 |

In [ ]:
import time

from travelling_salesman import (
    generate_cities,
    compute_distance_matrix,
    build_tsp_qubo,
    classical_brute_force,
    solve_with_qaoa,
    extract_best_tour,
    solve_partitioned_tsp,
    solve_with_pce,
    print_comparison,
    plot_cities,
    plot_comparison,
)

from divi.backends import MaestroSimulator

SHOTS = 20_000
SEED = 42

---

## Local example

### Part A — Direct QAOA (4 cities, 16 qubits)

In [ ]:
local_backend = MaestroSimulator(shots=SHOTS)
print("Using local MaestroSimulator")

N_CITIES_SMALL = 4
cities_small = generate_cities(N_CITIES_SMALL, seed=SEED)
dist_small = compute_distance_matrix(cities_small)
print(
    f"\n📍 Generated {N_CITIES_SMALL} cities ({N_CITIES_SMALL}² = {N_CITIES_SMALL**2} qubits)"
)

plot_cities(cities_small, title=f"TSP — {N_CITIES_SMALL} Cities (Direct QAOA)")

In [ ]:
bqm_small, _ = build_tsp_qubo(dist_small)
print(
    f"QUBO: {len(bqm_small.variables)} variables, {len(bqm_small.quadratic)} interactions"
)

classical_tour_small, classical_dist_small = classical_brute_force(dist_small)
print(
    f"Classical optimum: {classical_tour_small}  distance = {classical_dist_small:.4f}"
)

print("\n🚀 Running direct QAOA locally...")
t0 = time.time()
qaoa = solve_with_qaoa(
    bqm_small, n_layers=3, max_iterations=20, shots=SHOTS, backend=local_backend
)
result_small = extract_best_tour(qaoa, bqm_small, dist_small)
local_time_a = time.time() - t0

if result_small:
    q_tour_small, q_dist_small = result_small
    print(f"\n✅ QAOA tour: {q_tour_small}  distance = {q_dist_small:.4f}")
    print_comparison(
        classical_tour_small, classical_dist_small, q_tour_small, q_dist_small
    )

### Part C — PCE (Pauli Correlation Encoding, qubit compression)

In [ ]:
print(f"📍 Same {N_CITIES_SMALL}-city instance (with qubit compression)")

print("\n🚀 Running PCE-VQE locally...")
t0 = time.time()
pce_tour, pce_dist, pce_qubits = solve_with_pce(
    bqm_small,
    dist_small,
    n_layers=3,
    max_iterations=20,
    shots=SHOTS,
    backend=local_backend,
)
local_time_c = time.time() - t0

print(f"\n✅ PCE tour: {pce_tour}  distance = {pce_dist:.4f}")
print(f"   Qubits used: {pce_qubits}  (vs {N_CITIES_SMALL**2} for direct QAOA)")
print(f"\n⏱️ Phase 1 — Direct QAOA: {local_time_a:.1f}s, PCE: {local_time_c:.1f}s")

---

## Partitioned QAOA (8 cities, 64 qubits)

64-qubit QUBO — too large for direct QAOA. Divi's `PartitioningProgramEnsemble  # updated API` splits it into manageable sub-problems, and QoroService runs them in parallel.

**Requirements:**
Create a `.env` file in the repo root:
```
QORO_API_KEY="your_api_key_here"
```
The partitioned example also runs on the selected local backend.

In [ ]:
N_CITIES_LARGE = 8
cloud_backend = MaestroSimulator(shots=SHOTS)

cities_large = generate_cities(N_CITIES_LARGE, seed=SEED)
dist_large = compute_distance_matrix(cities_large)
print(f"📍 Generated {N_CITIES_LARGE} cities ({N_CITIES_LARGE**2} qubits)")
print(f"   ⚠️  64 qubits — too large for a single QAOA run")

plot_cities(cities_large, title=f"TSP — {N_CITIES_LARGE} Cities (Partitioned QAOA)")

bqm_large, _ = build_tsp_qubo(dist_large)
classical_tour_large, classical_dist_large = classical_brute_force(dist_large)
print(
    f"Classical optimum: {classical_tour_large}  distance = {classical_dist_large:.4f}"
)

print(f"\n☁️  Routing {N_CITIES_LARGE**2}-variable QUBO partitions to Qoro Maestro...")
t0 = time.time()
q_tour_large, q_dist_large = solve_partitioned_tsp(
    bqm_large,
    dist_large,
    decomposer_size=25,
    n_layers=2,
    max_iterations=10,
    shots=SHOTS,
    backend=cloud_backend,
)
cloud_time = time.time() - t0

print(f"\n✅ Partitioned QAOA tour: {q_tour_large}  distance = {q_dist_large:.4f}")
print_comparison(classical_tour_large, classical_dist_large, q_tour_large, q_dist_large)

plot_comparison(
    cities_large,
    classical_tour_large,
    classical_dist_large,
    q_tour_large,
    q_dist_large,
    save_path="tsp_partitioned_comparison.png",
)

print(
    f"\n⚡ Local  (Phase 1): {local_time_a + local_time_c:.1f}s for {N_CITIES_SMALL} cities"
)
print(f"Partitioned run: {cloud_time:.1f}s for {N_CITIES_LARGE} cities")

---

## Further exploration

Try changing the number of cities, partition size, optimizer settings, and shot count. Compare the repaired quantum tours with the classical reference.